In [1]:
# PyTorch supplies tensors, automatic differentiation, and GPU support.
import torch
# torch.nn contains neural-network layers and loss functions.
import torch.nn as nn
# torch.optim contains algorithms that update a model's parameters.
import torch.optim as optim

# MNIST is a dataset of 28 x 28 grayscale handwritten-digit images.
from torchvision.datasets import MNIST
# transforms provides preprocessing operations for images.
from torchvision import transforms
# DataLoader divides a dataset into manageable mini-batches.
from torch.utils.data import DataLoader

# Use an NVIDIA GPU through CUDA when one is available; otherwise use the CPU.
# The model and every input tensor must be placed on the same device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
# Convert each PIL image to a PyTorch tensor with shape [1, 28, 28].
# ToTensor also changes pixel values from integers in [0, 255]
# to floating-point values in [0.0, 1.0].
transform = transforms.ToTensor()

# Download/load the 60,000-image training portion of MNIST.
# transform is applied whenever an image is retrieved from the dataset.
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)

# Return 64 training examples at a time. Shuffling gives the model a new
# example order each epoch and helps prevent order-dependent learning.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Load the separate 10,000-image test set. The model never trains on it.
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

# Test order does not affect accuracy, so shuffling is unnecessary.
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Define a convolutional neural network by inheriting from nn.Module.
class SimpleCNN(nn.Module):
    def __init__(self):
        # Initialize the base nn.Module class so PyTorch can register and
        # manage all layers and trainable parameters in this model.
        super().__init__()

        # This section extracts spatial features from each input image.
        # An input batch has shape [batch_size, 1, 28, 28].
        self.features = nn.Sequential(
            # First convolution: 1 grayscale channel -> 8 feature maps.
            # A 3 x 3 filter scans the image. padding=1 preserves 28 x 28.
            nn.Conv2d(in_channels=1,
                       out_channels=8,
                       kernel_size=3,
                       padding=1),
            # Replace negative values with zero, introducing non-linearity.
            nn.ReLU(),
            # Keep the maximum from each 2 x 2 area. Spatial dimensions
            # become 14 x 14, so the shape is [batch, 8, 14, 14].
            nn.MaxPool2d(kernel_size=2),

            # Second convolution: combine 8 maps into 16 richer feature maps.
            # Padding again preserves the current height and width.
            nn.Conv2d(in_channels=8,
                       out_channels=16,
                      kernel_size=3,
                      padding=1),
            nn.ReLU(),
            # Reduce 14 x 14 to 7 x 7. Output: [batch, 16, 7, 7].
            nn.MaxPool2d(kernel_size=2)
        )

        # Fully connected layers use the extracted features to select a digit.
        self.classifier = nn.Sequential(
            # There are 16 * 7 * 7 = 784 features for each image.
            #output = input × weights + bias
            nn.Linear(in_features=16 * 7 * 7, out_features=64),
            nn.ReLU(),
            # Produce 10 raw class scores (logits), one for each digit 0-9.
            nn.Linear(in_features=64, out_features=10)
        )

    # forward describes how an input travels through the network.
    def forward(self, x):
        # [batch, 1, 28, 28] -> [batch, 16, 7, 7]
        x = self.features(x)
        # Flatten all dimensions except the batch dimension:
        # [batch, 16, 7, 7] -> [batch, 784].
        x = x.view(x.size(0), -1)
        # [batch, 784] -> [batch, 10]
        x = self.classifier(x)
        # Return logits. CrossEntropyLoss applies the required probability
        # operation internally, so a Softmax layer should not be added here.
        return x

# Construct the network and move all of its parameters to the chosen device.
model = SimpleCNN().to(device)

# Cross-entropy compares the 10 logits with the correct integer class label.
loss_fn = nn.CrossEntropyLoss()

# Adam adjusts every trainable weight using its calculated gradient.
# lr=0.001 controls the size of each update.
optimizer = optim.Adam(model.parameters(), lr=0.001)

# One epoch is one complete pass over all 60,000 training examples.
epochs = 5
for epoch in range(epochs):
    # Enable training behavior. This matters for layers such as dropout and
    # batch normalization, even though this particular network has neither.
    model.train()
    total_loss = 0.0

    # DataLoader supplies a batch of images and their correct digit labels.
    for images, labels in train_loader:
        # Move both inputs and targets to the same device as the model.
        images, labels = images.to(device), labels.to(device)

        # PyTorch accumulates gradients by default, so clear old gradients.
        optimizer.zero_grad()
        # Forward pass: calculate the model's 10 scores for every image.
        outputs = model(images)
        # Calculate how far the predictions are from the correct labels.
        loss = loss_fn(outputs, labels)
        # Backward pass: compute the gradient of the loss for every parameter.
        loss.backward()
        # Use the gradients to update the convolution and linear weights.
        optimizer.step()

        # loss.item() converts the one-value loss tensor to a Python number.
        total_loss += loss.item()

    # Report the mean loss across all mini-batches in this epoch.
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Switch to evaluation behavior before measuring performance on unseen data.
model.eval()
correct = 0
total = 0

# Gradients are unnecessary during testing. Disabling them reduces memory use
# and makes inference faster; optimizer.step() is never called here.
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # argmax selects the index of the largest score in each row. That
        # index is the predicted digit class from 0 through 9.
        predictions = outputs.argmax(dim=1)
        # Count the total number of evaluated examples.
        total += labels.size(0)
        # Comparison creates True for correct predictions. Sum counts them.
        correct += (predictions == labels).sum().item()

# Accuracy is the percentage of test examples classified correctly.
print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Save only the learned parameter dictionary, not the whole Python class.
# Reusing it later requires creating SimpleCNN again and loading these weights.
torch.save(model.state_dict(), "simple_cnn.pth")
print("Model saved to simple_cnn.pth")


Epoch [1/5], Loss: 0.3166
Epoch [2/5], Loss: 0.0916
Epoch [3/5], Loss: 0.0660
Epoch [4/5], Loss: 0.0526
Epoch [5/5], Loss: 0.0433
Test Accuracy: 98.77%
Model saved to simple_cnn.pth
